In [1]:
import matplotlib.pyplot as plt 
import numpy as np 
import pandas as pd 
from scipy import stats

### Table 4-22

In [2]:
df_t22 = pd.DataFrame({
    'treatment': np.repeat([1,2,3,4],4),
    'block': np.tile([1,2,3,4],4),
    'r_time': [73,74,np.nan,71,np.nan,75,67,72,73,75,68,np.nan,75,np.nan,72,75]
})

In [3]:
mask_t22 = ~df_t22['r_time'].isna()

In [4]:
df_t22['ave'] = df_t22['r_time'].mean()
df_t22['b_ave'] = df_t22.groupby('block')['r_time'].transform('mean')
df_t22['b_dev'] = df_t22['b_ave'] - df_t22['ave']
df_t22['t_ave'] = df_t22.groupby('treatment')['r_time'].transform('mean') 
df_t22['t_dev'] = df_t22['t_ave'] - df_t22['ave']

In [5]:
# sum of squares of block deviations (unadjusted)
(df_t22[mask_t22]['b_dev']**2).sum()

np.float64(55.0000000000001)

In [6]:
# sum of squares of treatment deviations (unadjusted)
(df_t22[mask_t22]['t_dev']**2).sum()

np.float64(11.666666666666705)

In [ ]:
# sum of squares of treatment deviations (adjusted)
((df_t22[mask_t22].groupby('treatment')[['r_time', 'b_ave']].transform('mean').diff(axis=1)['b_ave']*3)**2).sum()/8

np.float64(22.749999999999915)

In [ ]:
# sum of squares of block deviations (adjusted)
((df_t22[mask_t22].groupby('block')[['r_time', 't_ave']].transform('mean').diff(axis=1)['t_ave']*3)**2).sum()/8

np.float64(66.08333333333336)

In [ ]:
# Least squares estimation of parameters
df_t22_ls = pd.get_dummies(df_t22[mask_t22][['treatment', 'block', 'r_time']], columns=['treatment', 'block'], dtype=int)

In [42]:
X_22 = np.concat((np.ones((12,1)), df_t22_ls.iloc[:,-8:].to_numpy()), axis=1)
y_22 = df_t22_ls[['r_time']].to_numpy()
C_22 = np.array([
    [0,1,1,1,1,0,0,0,0],
    [0,0,0,0,0,1,1,1,1]
]) # constrains
X_22, y_22, C_22

(array([[1., 1., 0., 0., 0., 1., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 1., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0., 1.],
        [1., 0., 1., 0., 0., 0., 1., 0., 0.],
        [1., 0., 1., 0., 0., 0., 0., 1., 0.],
        [1., 0., 1., 0., 0., 0., 0., 0., 1.],
        [1., 0., 0., 1., 0., 1., 0., 0., 0.],
        [1., 0., 0., 1., 0., 0., 1., 0., 0.],
        [1., 0., 0., 1., 0., 0., 0., 1., 0.],
        [1., 0., 0., 0., 1., 1., 0., 0., 0.],
        [1., 0., 0., 0., 1., 0., 0., 1., 0.],
        [1., 0., 0., 0., 1., 0., 0., 0., 1.]]),
 array([[73.],
        [74.],
        [71.],
        [75.],
        [67.],
        [72.],
        [73.],
        [75.],
        [68.],
        [75.],
        [72.],
        [75.]]),
 array([[0, 1, 1, 1, 1, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 1, 1, 1, 1]]))

In [86]:
theta_22 = np.linalg.solve(
    
    np.block([
    [X_22.T@(X_22), C_22.T],
    [C_22, np.zeros((2,2))]]
),
np.concatenate((X_22.T@(y_22), np.array([[0],[0]])))
)
theta_22

array([[ 7.25000000e+01],
       [-1.12500000e+00],
       [-8.75000000e-01],
       [-5.00000000e-01],
       [ 2.50000000e+00],
       [ 8.75000000e-01],
       [ 3.00000000e+00],
       [-3.87500000e+00],
       [-3.64291930e-16],
       [-8.32667268e-17],
       [ 5.62371544e-32]])

In [59]:
np.block([
    [X_22.T@(X_22), C_22.T],
    [C_22, np.zeros((2,2))]]
)

array([[12.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,  0.,  0.],
       [ 3.,  3.,  0.,  0.,  0.,  1.,  1.,  0.,  1.,  1.,  0.],
       [ 3.,  0.,  3.,  0.,  0.,  0.,  1.,  1.,  1.,  1.,  0.],
       [ 3.,  0.,  0.,  3.,  0.,  1.,  1.,  1.,  0.,  1.,  0.],
       [ 3.,  0.,  0.,  0.,  3.,  1.,  0.,  1.,  1.,  1.,  0.],
       [ 3.,  1.,  0.,  1.,  1.,  3.,  0.,  0.,  0.,  0.,  1.],
       [ 3.,  1.,  1.,  1.,  0.,  0.,  3.,  0.,  0.,  0.,  1.],
       [ 3.,  0.,  1.,  1.,  1.,  0.,  0.,  3.,  0.,  0.,  1.],
       [ 3.,  1.,  1.,  0.,  1.,  0.,  0.,  0.,  3.,  0.,  1.],
       [ 0.,  1.,  1.,  1.,  1.,  0.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0.,  1.,  1.,  1.,  1.,  0.,  0.]])

In [64]:
np.concatenate((X_22.T@(y_22), np.array([[0],[0]])))

array([[870.],
       [218.],
       [214.],
       [216.],
       [222.],
       [221.],
       [224.],
       [207.],
       [218.],
       [  0.],
       [  0.]])

In [88]:
N_22 = df_t22[['treatment', 'block', 'r_time']].groupby(['treatment', 'block']).apply(pd.DataFrame).droplevel(2).unstack().isna().astype(int).to_numpy()
r_22 = 3
k_22 = 3

Ct_22 = r_22*np.eye(4) - (1/k_22)*N_22.dot(N_22.T)

theta_22[1:5].T.dot(Ct_22).dot(theta_22[1:5])

array([[22.75]])

### Problem 1
| Source | DF | SS | MS | F | P |
|-|-|-|-|-|-|
| Treatment | 4 | 1010.56 | 252.64 | 29.84 | 3.545e-8 |
| Block | 5 | 323.82 | 64.765 | 7.65 | 0.00037 |
| Error | 20 | 169.33 | 8.4665 |
| Total | 29 | 1503.71 |

In [99]:
stats.f.sf(29.84, dfn = 4, dfd=20), stats.f.sf(7.65, dfn = 5, dfd=20) # no significant difference btw treatments or blocks

(np.float64(3.5448475242810104e-08), np.float64(0.00036872054210170907))

### Problem 2
| Source | DF | SS | MS | F | P |
|-|-|-|-|-|-|
| Factor | 4 | 987.72 | 246.93 | 46.36 | 7.67e-9 |
| Block | 5 | 80.00 | 16 | 3.00 | 0.04 |
| Error | 20 | 106.52 | 5.326 |
| Total | 29 | 1174.24 |

In [106]:
stats.f.sf(46.36, dfn = 4, dfd=20), stats.f.sf(3, dfn = 5, dfd=20) # no significant difference btw treatments or blocks

(np.float64(7.666521780041871e-10), np.float64(0.035201337452666626))